# Treino da Luna em GPU

Este notebook treina o Piper `pt_BR-luna-medium` a partir do `luna-cloud.zip`
gerado por `scripts/build_luna_cloud_bundle.py`.

**Antes de rodar:** confira o `BUNDLE.json`. Se `trainable` for `false`, o
dataset tem menos de 60 s e o treino vai recusar — importe mais áudio da Luna
no Voice Lab e gere o bundle de novo. O recomendado é 10–30 min de fala.  
No Colab: **Runtime → Change runtime type → T4 GPU**, depois **Runtime →
Restart**.

In [ ]:
import sys, torch
print("Python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠ Ative a GPU: Runtime → Change runtime type → T4 GPU, depois reinicie.")

In [ ]:
# piper-tts 1.8.0 é a versão usada pelo dashboard; onnxscript é necessário
# para o export em torch >= 2.9.
!pip install -q "piper-tts[train]==1.8.0" onnxscript

In [ ]:
import json, pathlib, sys, zipfile

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

base = pathlib.Path("/content" if IN_COLAB else "/kaggle/working")
zip_candidates = [base / "luna-cloud.zip", pathlib.Path("/kaggle/working/luna-cloud.zip")]
zip_path = None

if IN_COLAB:
    print("Selecione luna-cloud.zip no seletor de arquivos...")
    uploaded = files.upload()
    if uploaded:
        zip_path = base / next(iter(uploaded))
if zip_path is None:
    zip_path = next((p for p in zip_candidates if p.exists()), None)
if zip_path is None and pathlib.Path("/kaggle/input").is_dir():
    matches = sorted(pathlib.Path("/kaggle/input").rglob("luna-cloud.zip"))
    zip_path = matches[0] if matches else None
if zip_path is None or not zip_path.exists():
    raise SystemExit(
        "luna-cloud.zip não encontrado. No Colab, faça upload; no Kaggle, "
        "copie o zip para /kaggle/working."
    )

with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(base)

bundle = base / "luna-cloud"
out = base / "luna-output"
print("bundle:", bundle)
if not (bundle / "BUNDLE.json").is_file():
    raise SystemExit(f"BUNDLE.json não encontrado em {bundle} — o zip está correto?")
meta = json.loads((bundle / "BUNDLE.json").read_text(encoding="utf-8"))
print(
    "dataset:", meta["clips"], "clipe(s),", f"{meta['seconds']:.2f}s",
    "| treinável:", meta["trainable"],
)
if not meta["trainable"]:
    print(
        "\n⚠ O script vai recusar: menos de 60 s de fala. Importe mais áudio "
        "da Luna, gere o bundle de novo e faça upload outra vez."
    )

In [ ]:
import sys

!{sys.executable} {bundle}/train_luna.py \
  --bundle {bundle} \
  --out {out} \
  --epochs 150 \
  --batch-size 16

In [ ]:
result = out / "pt_BR-luna-medium-cloud.zip"
if not result.is_file():
    raise SystemExit(
        f"{result} não foi gerado. Veja o erro da célula de treino — o mais "
        "comum é dataset abaixo de 60 s ou GPU desligada."
    )
print("resultado:", result, f"({result.stat().st_size / 1024 / 1024:.1f} MB)")
if IN_COLAB:
    files.download(str(result))
else:
    print("Baixe o zip pelo painel Output do Kaggle.")

## Depois do download

Na máquina do Wallie, anexe o par `.onnx` + `.onnx.json` e ative o perfil:

```bash
python scripts/attach_cloud_voice.py baixados/pt_BR-luna-medium-cloud.zip \
  --profile luna --activate
```

O helper recusa qualquer modelo treinado com menos de 60 s. Reinicie o
dashboard depois para descartar a configuração antiga em memória.